# User Coherence Evaluation on Foundry Agent Traces

Evaluates user coherence (derail detection) on multi-turn conversations fetched from Azure AI Foundry agents:
- **legal-contract-review-sim** (20 conversations)
- **airline-customer-service-sim** (30 conversations)

Each conversation is evaluated independently, producing per-step derail scores and an aggregate mean score.

In [ ]:
import json
import os
import asyncio
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# All files (datasets, .env, results) live in the same directory as this notebook
load_dotenv()

# Import evaluator and trace normalizer
from azure.ai.evaluation._evaluators._user_coherence._user_coherence import (
    UserCoherenceEvaluator,
    normalize_trace_messages,
)

# Load datasets (produced by fetch_agent_conversations.py in the same folder)
AGENTS = os.environ.get("AGENTS", "legal-contract-review-sim,airline-customer-service-sim").split(",")

datasets = {}
for agent_name in AGENTS:
    filename = f"{agent_name}_conversations.json"
    if not Path(filename).exists():
        print(f"WARNING: {filename} not found — run fetch_agent_conversations.py first")
        continue
    with open(filename, encoding="utf-8") as f:
        datasets[agent_name] = json.load(f)
    print(f"Loaded {len(datasets[agent_name])} conversations for {agent_name}")

Loaded 20 conversations for legal-contract-review-sim
Loaded 30 conversations for airline-customer-service-sim


## Preview: Normalize Trace Messages

The raw trace messages from Foundry use `output_text`/`input_text` content types and include extra metadata. `normalize_trace_messages()` converts them to simple `{"role": ..., "content": ...}` dicts.

In [7]:
# Preview normalization on the first conversation of each agent
# NOTE: Foundry conversations.items.list() returns messages in reverse chronological order (newest first),
# so we reverse them to get the correct conversation flow.
for agent_name, convs in datasets.items():
    raw_msgs = convs[0]["messages"]
    # Reverse to chronological order, then normalize
    normalized = normalize_trace_messages(list(reversed(raw_msgs)))
    print(f"\n=== {agent_name} (conversation {convs[0]['conversation_id']}) ===")
    print(f"Raw messages: {len(raw_msgs)}, Normalized: {len(normalized)}")
    for msg in normalized:
        print(f"  [{msg['role']}] {msg['content']}...")


=== legal-contract-review-sim (conversation conv_f77aa9489231d2c000qnNW5b8ksh2XPCbH7JIWhjL6uMr4XGVE) ===
Raw messages: 10, Normalized: 10
  [user] Hey, this is Omar Salim. Quick one — is the limitation of liability clause in this SaaS contract safe to sign? I need to get it back today....
  [assistant] Thanks, Omar. To give you a proper recommendation, I'll need to:

1. Check the contract metadata (so I know the jurisdiction and relevant policies).
2. Find and read the actual limitation of liability clause.

I’ll get started now.

[TOOL CALL] get_contract_metadata({"contract_id": "saas-2024-omar"})

[TOOL RESULT] {
  "parties": ["Salim Enterprises LLC", "CloudLogic Inc."],
  "effective_date": "2024-06-10",
  "governing_law": "New York",
  "jurisdiction": "New York",
  "contract_type": "SaaS Master Agreement",
  "page_count": 22
}

[TOOL CALL] list_clauses({"contract_id": "saas-2024-omar"})

[TOOL RESULT] {
  "clauses": [
    {"clause_id": "1.1", "heading": "Definitions", "page": 1},
 

## Initialize UserCoherenceEvaluator

Configure the evaluator with an Azure OpenAI model deployment.

In [ ]:
# Read Azure OpenAI config from environment (loaded from .env above)
AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")

print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Deployment: {AZURE_OPENAI_DEPLOYMENT}")
print(f"API version: {AZURE_OPENAI_API_VERSION}")
print(f"API key: {'***' + AZURE_OPENAI_API_KEY[-4:] if AZURE_OPENAI_API_KEY else '(not set)'}")

In [ ]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_DEPLOYMENT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

credential = DefaultAzureCredential()

evaluator = UserCoherenceEvaluator(model_config=model_config, credential=credential)
print("UserCoherenceEvaluator initialized")

Class UserCoherenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


UserCoherenceEvaluator initialized


## Run Evaluation Per Conversation

For each agent, evaluate every conversation independently and collect results.

In [10]:
import time

all_results = {}  # agent_name -> list of result dicts

for agent_name, convs in datasets.items():
    print(f"\n{'='*80}")
    print(f"Evaluating: {agent_name} ({len(convs)} conversations)")
    print(f"{'='*80}")

    agent_results = []

    for i, conv in enumerate(convs):
        conv_id = conv["conversation_id"]
        raw_messages = conv["messages"]

        # Reverse to chronological order (API returns newest-first), then normalize
        messages = normalize_trace_messages(list(reversed(raw_messages)))

        if len(messages) < 2:
            print(f"  [{i+1}/{len(convs)}] {conv_id[:30]}... SKIPPED (only {len(messages)} messages)")
            continue

        print(f"  [{i+1}/{len(convs)}] {conv_id[:30]}... ({len(messages)} messages)", end="")

        try:
            # Evaluate using the conversation input format
            result = evaluator(conversation={"messages": messages})
            score = result.get("user_coherence", float("nan"))
            per_step = result.get("user_coherence_per_step", [])
            print(f" -> score={score:.2f} ({len(per_step)} steps)")

            agent_results.append({
                "conversation_id": conv_id,
                "agent_name": agent_name,
                "num_messages": len(messages),
                "user_coherence": score,
                "per_step_scores": per_step,
                "total_tokens": result.get("user_coherence_total_tokens", 0),
                "model": result.get("user_coherence_model", ""),
            })
        except Exception as e:
            print(f" -> ERROR: {e}")
            agent_results.append({
                "conversation_id": conv_id,
                "agent_name": agent_name,
                "num_messages": len(messages),
                "user_coherence": float("nan"),
                "error": str(e),
            })

        # Small delay to avoid rate limiting
        time.sleep(0.5)

    all_results[agent_name] = agent_results
    print(f"\n  Completed {len(agent_results)} evaluations for {agent_name}")


Evaluating: legal-contract-review-sim (20 conversations)
  [1/20] conv_f77aa9489231d2c000qnNW5b8... (10 messages) -> score=2.00 (5 steps)
  [2/20] conv_ec319379ae2a86f200K0O0Fkh... (12 messages) -> score=2.00 (6 steps)
  [3/20] conv_b09635d5925bbf7a00FRGdvPZ... (12 messages) -> score=2.00 (6 steps)
  [4/20] conv_7b72c23198af6b5700N8yPmku... (10 messages) -> score=2.00 (5 steps)
  [5/20] conv_548358c9f3e8b85700xZukrtt... (13 messages) -> score=2.00 (6 steps)
  [6/20] conv_bb91bfbd599f7a2400w14I3u8... (10 messages) -> score=2.00 (5 steps)
  [7/20] conv_8685469b8f0655ea00e9DPuaJ... (12 messages) -> score=2.00 (6 steps)
  [8/20] conv_52dd8ae56c9c08d200zpbeHfC... (13 messages) -> score=2.00 (6 steps)
  [9/20] conv_a9b88a9fdc7891cc00C6i1N2Q... (10 messages) -> score=2.00 (5 steps)
  [10/20] conv_83ecc78a59e5184500fhb3PDR... (13 messages) -> score=2.00 (6 steps)
  [11/20] conv_cd82c5f00ab03d4f00T5mKPUK... (10 messages) -> score=2.00 (5 steps)
  [12/20] conv_f416e980089e4461004Kfd229... (12 m

## Results Summary

Aggregate results per agent and display a summary table.

In [11]:
import math

# Build summary DataFrames
for agent_name, results in all_results.items():
    print(f"\n{'='*60}")
    print(f"Agent: {agent_name}")
    print(f"{'='*60}")

    df = pd.DataFrame([
        {
            "conversation_id": r["conversation_id"][:30] + "...",
            "messages": r["num_messages"],
            "user_coherence": r["user_coherence"],
            "tokens": r.get("total_tokens", 0),
        }
        for r in results
    ])

    display(df)

    valid_scores = [r["user_coherence"] for r in results if not math.isnan(r["user_coherence"])]
    if valid_scores:
        print(f"\n  Conversations evaluated: {len(valid_scores)}")
        print(f"  Mean user_coherence: {sum(valid_scores)/len(valid_scores):.3f}")
        print(f"  Min: {min(valid_scores):.2f}, Max: {max(valid_scores):.2f}")
        print(f"  Std: {pd.Series(valid_scores).std():.3f}")
    else:
        print("  No valid scores")


Agent: legal-contract-review-sim


,conversation_id,messages,user_coherence,tokens
0,conv_f77aa9489231d2c000qnNW5b8...,10,2.0,6112
1,conv_ec319379ae2a86f200K0O0Fkh...,12,2.0,4376
2,conv_b09635d5925bbf7a00FRGdvPZ...,12,2.0,4716
3,conv_7b72c23198af6b5700N8yPmku...,10,2.0,4444
4,conv_548358c9f3e8b85700xZukrtt...,13,2.0,5171
5,conv_bb91bfbd599f7a2400w14I3u8...,10,2.0,4420
6,conv_8685469b8f0655ea00e9DPuaJ...,12,2.0,5525
7,conv_52dd8ae56c9c08d200zpbeHfC...,13,2.0,7766
8,conv_a9b88a9fdc7891cc00C6i1N2Q...,10,2.0,10250
9,conv_83ecc78a59e5184500fhb3PDR...,13,2.0,9542



  Conversations evaluated: 20
  Mean user_coherence: 2.000
  Min: 2.00, Max: 2.00
  Std: 0.000

Agent: airline-customer-service-sim


,conversation_id,messages,user_coherence,tokens
0,conv_90b677c1b52c2aaa002Jy9lpF...,8,2.0,3739
1,conv_83ed527d93b557e700Pq8X0C5...,8,2.0,3735
2,conv_8decba6f597c748a004Ojo7SI...,8,2.0,3399
3,conv_96d221abc4d8f36300oedrXN6...,14,2.0,4700
4,conv_fba1689acf4a9b9700JKnZzD0...,8,2.0,3143
5,conv_05de0be8304ea52e00KzdHT8L...,14,2.0,4164
6,conv_c25a99a749b6b69000V3brZuW...,14,2.0,4111
7,conv_aaa3bb18c933123d00iSLM5Cs...,8,2.0,3304
8,conv_1ba8c189be52254600XovYN3G...,8,2.0,3262
9,conv_8345462fadd6dc6600cRMRSmd...,10,2.0,3780



  Conversations evaluated: 30
  Mean user_coherence: 1.983
  Min: 1.50, Max: 2.00
  Std: 0.091


## Per-Step Score Distribution

Visualize the distribution of per-step derail scores across all conversations for each agent.

In [13]:
from collections import Counter

for agent_name, results in all_results.items():
    print(f"\n{'='*60}")
    print(f"Agent: {agent_name} — Per-Step Score Distribution")
    print(f"{'='*60}")

    score_counts = Counter()
    total_steps = 0
    derailed_conversations = []

    for r in results:
        per_step = r.get("per_step_scores", [])
        for step in per_step:
            s = step.get("derail_score")
            if s is not None:
                score_counts[s] += 1
                total_steps += 1

        # Flag conversations with any score=0 (derailed)
        zeros = [step for step in per_step if step.get("derail_score") == 0]
        if zeros:
            derailed_conversations.append({
                "conversation_id": r["conversation_id"][:30] + "...",
                "derailed_steps": [
                    {"step": z.get("conversation_step"), "explanation": z.get("explanation", "")}
                    for z in zeros
                ],
            })

    print(f"\n  Total scored steps: {total_steps}")
    for score in [2, 1, 0]:
        count = score_counts.get(score, 0)
        pct = (count / total_steps * 100) if total_steps else 0
        bar = "█" * int(pct / 2)
        label = {2: "Likely/logical", 1: "Weak/possible", 0: "Not logical"}[score]
        print(f"  Score {score} ({label}): {count:3d} ({pct:5.1f}%) {bar}")

    if derailed_conversations:
        print(f"\n  Conversations with derailed steps (score=0):")
        for dc in derailed_conversations:
            print(f"    {dc['conversation_id']}")
            for ds in dc["derailed_steps"]:
                print(f"      Step {ds['step']}: {ds['explanation']}")


Agent: legal-contract-review-sim — Per-Step Score Distribution

  Total scored steps: 85
  Score 2 (Likely/logical):  85 (100.0%) ██████████████████████████████████████████████████
  Score 1 (Weak/possible):   0 (  0.0%) 
  Score 0 (Not logical):   0 (  0.0%) 

Agent: airline-customer-service-sim — Per-Step Score Distribution

  Total scored steps: 110
  Score 2 (Likely/logical): 109 ( 99.1%) █████████████████████████████████████████████████
  Score 1 (Weak/possible):   0 (  0.0%) 
  Score 0 (Not logical):   1 (  0.9%) 

  Conversations with derailed steps (score=0):
    conv_1b255d874727afda00DY7bcSJ...
      Step 3: The user repeats the exact same information and request as in the previous step, which does not logically follow after the agent already addressed these details.


## Save Results

Save the full evaluation results (including per-step details) to JSON files.

In [ ]:
for agent_name, results in all_results.items():
    output_file = f"{agent_name}_user_coherence_results.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Saved {len(results)} results to {output_file}")

Saved 20 results to legal-contract-review-sim_user_coherence_results.json
Saved 30 results to airline-customer-service-sim_user_coherence_results.json
